## PWR Assembly Branching

In this example, we will extend the functionality described in pwr_assembly_depletion.ipynb by using a structured branching framework to explore the parameter space of an assembly.

We can begin by importing all the tools we will need from Scarabée.

In [ ]:
from scarabee import (
    NDLibrary,
    MaterialComposition,
    Material,
    Fraction,
    DensityUnits,
)
from scarabee.reseau import FuelPin, GuideTube, PWRAssembly, Symmetry, PWRCaseMatrix
import numpy as np
import matplotlib.pyplot as plt

As before, we can setup our assembly

In [ ]:
ndl = NDLibrary()

# Define Fuel at 3.1 w/o enrichment
Fuel31Comp = MaterialComposition(Fraction.Atoms, name="Fuel 3.1%")
Fuel31Comp.add_leu(3.1, 1.0) # The add_leu method assumes weight enrichment !
Fuel31Comp.add_element("O", 2.0)
Fuel31 = Material(Fuel31Comp, 900.0, 10.30166, DensityUnits.g_cm3, ndl)

# Define the cladding
CladComp = MaterialComposition(Fraction.Weight, name="Zircaloy 4")
CladComp.add_element("O", 0.00125)
CladComp.add_element("Cr", 0.0010)
CladComp.add_element("Fe", 0.0021)
CladComp.add_element("Zr", 0.98115)
CladComp.add_element("Sn", 0.0145)
Clad = Material(CladComp, 575.0, 6.55, DensityUnits.g_cm3, ndl)

# Define the helium gas
HeComp = MaterialComposition(Fraction.Atoms, name="He Gas")
HeComp.add_element("He", 1.0)
He = Material(HeComp, 575.0, 0.0015981, DensityUnits.g_cm3, ndl)

# Create a fuel pin
fp = FuelPin(
    fuel=Fuel31,
    fuel_radius=0.39218,
    gap=He,
    gap_radius=0.40005,
    clad=Clad,
    clad_radius=0.45720,
)

# Create a guide tube
gt = GuideTube(inner_radius=0.56134, outer_radius=0.60198, clad=Clad)
"""

#         |--- Half pin cells
#         v
cells = [[fp, fp, fp, fp, fp, fp, fp, fp, fp],
         [fp, fp, fp, fp, fp, fp, fp, fp, fp],
         [gt, fp, fp, gt, fp, fp, fp, fp, fp],
         [fp, fp, fp, fp, fp, gt, fp, fp, fp],
         [fp, fp, fp, fp, fp, fp, fp, fp, fp],
         [gt, fp, fp, gt, fp, fp, gt, fp, fp],
         [fp, fp, fp, fp, fp, fp, fp, fp, fp],
         [fp, fp, fp, fp, fp, fp, fp, fp, fp],
         [gt, fp, fp, gt, fp, fp, gt, fp, fp]] # <- Half pin cells

asmbly = PWRAssembly(
    pitch=1.25984,
    assembly_pitch=21.50364,
    shape=(17, 17),
    symmetry=Symmetry.Quarter,
    moderator={
        "temperature": 575.0,
        "pressure": 15.5132,
        "boron-ppm": 975.0,
    },
    cells=cells,
    ndl=ndl,
)
"""

# For my personal computer I ran a smaller problem
cells = [
    [fp, fp, fp, fp, fp],
    [gt, fp, gt, fp, fp],
    [gt, fp, gt, gt, fp],
    [fp, fp, fp, fp, fp],
    [gt, fp, gt, gt, fp],
]

asmbly = PWRAssembly(
    pitch=1.25984,
    shape=(9, 9),
    symmetry=Symmetry.Quarter,
    moderator={
        "temperature": 575.0,
        "pressure": 15.5132,
        "boron-ppm": 975.0,
    },
    cells=cells,
    ndl=ndl,
)

asmbly.moc_num_angles = 16
asmbly.moc_track_spacing = 0.1
asmbly.corrector_transport = False

We could call `asmbly.solve()` now to run the single k-eigenvalue problem as before

Instead, we pass a `PWRCaseMatrix` to the assembly, defining what parameters to 'branch' over in our calculations

In [ ]:
# Every parameter into the PWRCaseMatrix is optional
# Default parameters for the branch flags are false

options = PWRCaseMatrix(
    boron_values=[0.0, 800.0, 1600.0],
    branch_boron=True, 
    moderator_temps=[560.0, 580.0, 600.0], # Do not need to set the remaining parameters, just for show
    branch_moderator_temp=False,
    moderator_pressures=[15.0, 16.0],
    branch_moderator_pressure=False
)
asmbly.case_matrix = options

There is also the option to add depletion back in, configured through the main `PWRAssembly`

In [ ]:
asmbly.depletion_exposure_steps = np.array(5*[0.01] + [0.15] + [0.8] + [1.] + 15*[2.])

And now we can solve using `asmbly.run_branches()`. This produces an `AssemblySlice` which contains an `AssemblyStatePoint` object. Currently this just stores a `DiffusionData` object, but can be extended to include more neutronic information

In [ ]:
asmbly.run_branches()
asmbly_slice = asmbly.assembly_slice
print(f"\n{asmbly_slice}")

Now we receive a range of results and can plot them as desired

In [ ]:
print("\nAbsorption Cross Section (Burnup x Boron):")
header = f"{'Burnup (MWd/kg)':>16} |"
for boron in options.boron_values:
    header += f" {boron:>7.0f} ppm |"
print(header)
print("-" * 70)

for burnup_idx in range(asmbly_slice.num_burnup_steps):
    exposure = asmbly_slice.exposures[burnup_idx]
    row = f"{exposure:>16.2f} |"
    
    for boron_idx in range(asmbly_slice.num_boron_values):
        state = asmbly_slice.get_state(burnup_step=burnup_idx, boron_idx=boron_idx)
        if state is not None:
            row += f" {state.diffusion_data.xs.Ea(1):>11.5f} |"
        else:
            row += f" {'MISSING':>11} |"
    
    print(row)

print("=" * 70)

In [ ]:
Sigma_a = np.zeros((len(asmbly.exposures), len(options.boron_values)))

for i in range(len(asmbly.exposures)):
    for j in range(len(options.boron_values)):

        sp = asmbly.assembly_slice.get_state(burnup_step=i, boron_idx=j).diffusion_data.xs.Ea(1)
        Sigma_a[i, j] = sp

plt.figure(figsize=(7,5))
plt.pcolormesh(
    options.boron_values,
    asmbly.exposures,
    Sigma_a,
    shading='nearest',
    cmap="Oranges_r"
)
plt.colorbar(label=r"$\Sigma_a$")

plt.xlabel("Boron concentration (ppm)")
plt.ylabel("Exposure (MWd/kg)")
plt.xlim(0, 1600)
plt.ylim(0, 33)
plt.tight_layout()
plt.show()


This has shown branching over exposure and boron concentration, but it can be performed over moderator temperature and pressure at the same time as well, though the computational complexity rises quickly. 

Future plans involve changing the current branching format (solving every permutation of parameter values, resulting in ixjxkxl number of runs) to instead use depletion 'spines' and a more industry aligned case matrix, as outlined in the Handbook of Nuclear Engineering, Chapter 9, Section 7.

Additionally, the `PWRCaseMatrix` could in future be extended to cover more parameters, such as control rod insertion.